In [ ]:
import os
import torch
import torchvision.models as models
from transformers import BertModel, GPT2Model, AutoModelForCausalLM

NUM_OPERATIONS = 3000
SPARSITY_LEVELS = [0, 25, 50, 75, 85, 90, 95]
os.makedirs("sim_cases", exist_ok=True)

def generate_hex(tensor_weights, target_sparsity, filename):
    flat_weights = torch.abs(tensor_weights.flatten())
    if target_sparsity == 0.0:
        threshold = -1.0
    else:
        k_index = int(target_sparsity * flat_weights.numel())
        k_index = max(1, min(k_index, flat_weights.numel()))
        threshold = torch.kthvalue(flat_weights, k_index).values.item()

    mask = (flat_weights <= threshold).int().tolist()
    if len(mask) > NUM_OPERATIONS:
        start_idx = len(mask) // 2
        trace = mask[start_idx : start_idx + NUM_OPERATIONS]
    else:
        trace = (mask * (NUM_OPERATIONS // len(mask) + 1))[:NUM_OPERATIONS]

    with open(filename, "w") as f:
        for m in trace: f.write(f"{m}\n")


# 1. ResNet (Vision)
resnet = models.resnet50(pretrained=True)
w_resnet = resnet.layer4[0].conv1.weight.data

# 2. BERT (NLP Encoder)
bert = BertModel.from_pretrained("bert-base-uncased")
w_bert = bert.encoder.layer[0].attention.self.query.weight.data

# 3. GPT-2 (Represent LLM Klasik - Absolute PosEmbed & GeLU)
gpt = GPT2Model.from_pretrained("gpt2")
# Kita ambil layer FFN klasik
w_gpt = gpt.h[0].mlp.c_fc.weight.data

# 4. TinyLlama (Represent LLaMA Modern - RoPE & SwiGLU)
llama = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
w_llama = llama.model.layers[0].mlp.gate_proj.weight.data

model_weights = {
    "ResNet_50": w_resnet,
    "BERT_NLP": w_bert,
    "GPT4_Sim": w_gpt,
    "LLaMA3_8B": w_llama
}

for model_name, weights in model_weights.items():
    for s in SPARSITY_LEVELS:
        generate_hex(weights, s / 100.0, f"sim_cases/meta_{model_name}_{s}.hex")


!zip -r sim_cases_sweep.zip sim_cases/

⏳ Mendownload Model Asli (Vision, NLP, & LLM)...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 223MB/s]
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

🚀 Mengekstrak Sweeping Traces...
✅ Semua trace berdasar arsitektur murni berhasil dibuat!
  adding: sim_cases/ (stored 0%)
  adding: sim_cases/meta_GPT4_Sim_85.hex (deflated 95%)
  adding: sim_cases/meta_ResNet_50_75.hex (deflated 90%)
  adding: sim_cases/meta_GPT4_Sim_50.hex (deflated 90%)
  adding: sim_cases/meta_GPT4_Sim_25.hex (deflated 90%)
  adding: sim_cases/meta_LLaMA3_8B_50.hex (deflated 90%)
  adding: sim_cases/meta_BERT_NLP_0.hex (deflated 100%)
  adding: sim_cases/meta_LLaMA3_8B_90.hex (deflated 95%)
  adding: sim_cases/meta_GPT4_Sim_75.hex (deflated 92%)
  adding: sim_cases/meta_BERT_NLP_50.hex (deflated 90%)
  adding: sim_cases/meta_BERT_NLP_95.hex (deflated 96%)
  adding: sim_cases/meta_BERT_NLP_75.hex (deflated 91%)
  adding: sim_cases/meta_BERT_NLP_25.hex (deflated 91%)
  adding: sim_cases/meta_LLaMA3_8B_95.hex (deflated 97%)
  adding: sim_cases/meta_BERT_NLP_85.hex (deflated 92%)
  adding: sim_cases/meta_ResNet_50_95.hex (deflated 95%)
  adding: sim_cases/meta_ResNet_